# Edu Nexus – Phase 1: PPT/PPTX to DOCX Converter

This notebook handles text-based PowerPoint files.
It extracts text from all slides using python-pptx and
converts the content into a standardized DOCX file.

OCR is NOT used in this step.


In [2]:
!pip install python-pptx

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:


from pptx import Presentation
from pathlib import Path
from typing import List


In [5]:
# ===== PPT TEXT EXTRACTION =====

def extract_text_from_ppt(ppt_path: Path) -> List[str]:
    """
    Extracts text from a PPT/PPTX file slide-by-slide.
    Returns a list of text blocks in order.
    """
    presentation = Presentation(ppt_path)
    extracted_blocks = []

    for slide_idx, slide in enumerate(presentation.slides, start=1):
        slide_text = []
        for shape in slide.shapes:
            if shape.has_text_frame:
                slide_text.append(shape.text.strip())

        if slide_text:
            extracted_blocks.append(
                f"Slide {slide_idx}:\n" + "\n".join(slide_text)
            )

    return extracted_blocks


In [6]:
# ===== PPT TO DOCX PIPELINE =====

from docx import Document
from docx.shared import Pt

def convert_ppt_to_docx(ppt_path: Path, output_path: Path):
    """
    Converts a PPT/PPTX file into a DOCX document.
    """
    text_blocks = extract_text_from_ppt(ppt_path)

    if not text_blocks:
        raise ValueError("No extractable text found in PPT.")

    full_text = "\n\n".join(text_blocks)

    # Reuse logic from writer (inline to keep notebook independent)
    doc = Document()
    doc.add_heading(ppt_path.stem, level=1)

    for line in full_text.split("\n"):
        para = doc.add_paragraph(line)
        for run in para.runs:
            run.font.size = Pt(11)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(output_path)

    return output_path


In [7]:
# ===== TEST PPT CONVERSION =====

ppt_input = Path("../../edu_nexus_db/raw/ppt")
ppt_files = list(ppt_input.glob("*.ppt*"))

if not ppt_files:
    raise FileNotFoundError("No PPT files found in raw/ppt folder.")

ppt_file = ppt_files[0]

output_docx = Path("../../edu_nexus_db/normalized/docx") / f"{ppt_file.stem}.docx"

convert_ppt_to_docx(ppt_file, output_docx)

print("PPT converted to DOCX:", output_docx.resolve())


PPT converted to DOCX: C:\Users\kulva\Desktop\Minor Project\edu_nexus_db\normalized\docx\Session 11 - Basics of Inheritance.docx
